In [5]:
# 代码作用：读取LIMS数据汇总为 精糖指标合格标准数据表
# 核心逻辑：数据预处理，包括修改字段名称、填充空值、类型转换、派生日期字段
# 创建时间：2026/04/03
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType, DateType, DoubleType
from pyspark.sql.functions import (
    current_timestamp, split, lit, col, row_number, sum as spark_sum,
    round as spark_round, avg, when, to_date, count, coalesce,
    regexp_extract, concat_ws, lpad
)
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()

# ===================== 1. 读取LIMS样品分析精糖指标合格标准数据 =====================
lims_jtzb_sql = """
SELECT 
    T1.orgcode AS org_code,
    T2.productioncategory AS production_category,
    T1.zj AS crushing_season,
    T1.dismonth,
    T2.wl_name AS material_name,
    T2.name1 AS item_name,
    T2.xxz AS lower_limit,
    T2.sxz AS upper_limit,
    std_value AS standard_value,
    T1.create_time,
    T1.update_time,
    T1.create_name AS created_by
FROM ods.ods_lims_bzjx_prod_item_std_month_b T1
LEFT JOIN ods.ods_lims_bzjx_prod_item_std_month_detail_b_f_1h T2 
    ON T1.id = T2.pid
"""
lims_jtzb_df = spark.sql(lims_jtzb_sql)

# ===================== 2. 数据预处理 =====================
# 2.1 将 lower_limit 和 upper_limit 转换为 Decimal(10,2) 类型，空值填充 0
# 注意：原始数据可能是字符串或空值，先 cast 为 Decimal，再用 coalesce 替换 null 为 0
decimal_type = DecimalType(10, 2)
lims_jtzb_df = lims_jtzb_df.withColumn(
    "lower_limit",
    coalesce(col("lower_limit").cast(decimal_type), lit(0).cast(decimal_type))
).withColumn(
    "upper_limit",
    coalesce(col("upper_limit").cast(decimal_type), lit(0).cast(decimal_type))
)

# 2.2 基于 dismonth（格式如 "2026年4月"）生成日期字段 dismonth_date（当月第一天）
# 提取年份（4位数字）和月份（1-2位数字），拼接成 "yyyy-MM-01" 后转为 DateType
lims_jtzb_df = lims_jtzb_df.withColumn(
    "dismonth_date",
    to_date(
        concat_ws("-",
            regexp_extract(col("dismonth"), "(\\d{4})", 1),          # 年份
            lpad(regexp_extract(col("dismonth"), "(\\d{1,2})月", 1), 2, "0"),  # 月份，补零
            lit("01")                                                 # 固定日
        ),
        "yyyy-MM-dd"
    )
)

# 查看预处理后的数据（可选）
lims_jtzb_df.show(10, truncate=False)

# ===================== 3. 写入 DWD 层表 =====================
target_table = "dwd.dwd_lims_sugar_quality_standard_f_1h"
lims_jtzb_df.write.mode("overwrite").saveAsTable(target_table)
print(f"数据已成功写入表: {target_table}，记录数: {lims_jtzb_df.count()}")

Spark 启动中...
+--------+-------------------+---------------+----------+-------------+---------+-----------+-----------+--------------+-------------------+-------------------+----------+-------------+
|org_code|production_category|crushing_season|dismonth  |material_name|item_name|lower_limit|upper_limit|standard_value|create_time        |update_time        |created_by|dismonth_date|
+--------+-------------------+---------------+----------+-------------+---------+-----------+-----------+--------------+-------------------+-------------------+----------+-------------+
|FNR     |炼糖生产           |24/25          |2024年12月|R3膏         |锤度     |87.00      |91.00      |87～91        |2024-12-01 08:00:00|2024-12-30 09:12:52|韦愉谢    |2024-12-01   |
|FNR     |炼糖生产           |24/25          |2024年12月|R2膏         |锤度     |87.00      |91.00      |87～91        |2024-12-01 08:00:00|2024-12-30 09:12:52|韦愉谢    |2024-12-01   |
|FNR     |炼糖生产           |24/25          |2024年12月|R1膏         |锤度     |87.00      